shared_root <- Sys.getenv("REGA_SHARED_ROOT", unset = normalizePath(".."))
separate_root <- Sys.getenv("REGA_SEPARATE_ROOT")
if (!nzchar(separate_root)) stop("Set REGA_SEPARATE_ROOT")
# Cross-cell-type module gene-composition bubble plot

Selected modules: **AG_M13, EC_M14, EC_M17, MTC_M6, MTC_M13, PFC_M14, PFC_M18**.

- Rows = cell types
- Columns = region_module
- Bubble area = fraction of module genes from that cell type
- Bubble color = cell type
- Number inside bubble = `n_genes`
- X-axis `(n=...)` = `module_total` genes
- Colored strip above columns = brain region

The workflow checks that gene counts sum to `module_total` and fractions sum to approximately 1 for each module.


In [ ]:
# ============================================================
# Cross-cell-type module gene composition bubble plot
#
# Rows    = cell type
# Columns = selected region_module
# Bubble area = fraction of module genes from that cell type
# Bubble color = cell type
# Number inside bubble = n_genes
# X-axis label n = module_total genes
# Top strip = brain region color
# ============================================================

shared_root <- normalizePath(Sys.getenv("REGA_SHARED_ROOT", unset = normalizePath("..")))

suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(ggplot2)
  library(scales)
})

# ============================================================


In [ ]:
# 1. User settings
# ============================================================

INPUT_DIR <- file.path(shared_root, "part3_process_REGA_output/11module_celltype_fraction_tables")

#OUT_DIR <- file.path(INPUT_DIR, "selected_cross_celltype_modules_bubbleplot")
OUT_DIR <- file.path(shared_root, "part3_process_REGA_output/figs/module_fraction_bubble_plot")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SELECTED_MODULES <- tibble(region_module = readLines(file.path(shared_root, "part2_run_REGA/08pathology_tests/selected_module.txt"))) %>%
  mutate(region = sub("_M[0-9]+$", "", region_module),
         module = sub("^[^_]+_", "", region_module))

MODULE_ORDER <- SELECTED_MODULES$region_module
CELLTYPE_ORDER <- c("Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune")

REGION_COLORS <- c(
  AG  = "#8DA0CB",
  EC  = "#FC8D62",
  HC  = "#66C2A5",
  MTC = "#E78AC3",
  PFC = "#B3B3B3",
  TH  = "#A6D854"
)

CELLTYPE_COLORS <- c(
  Exc        = "#0072B2",
  Inh        = "#E69F00",
  Ast        = "#009E73",
  Oli        = "#D55E00",
  OPC        = "#CC79A7",
  Mic_Immune = "#6B5B53"
)

# ============================================================


In [ ]:
# 2. Read only the regions needed for the selected modules
# ============================================================

regions_needed <- unique(SELECTED_MODULES$region)

read_one_region <- function(region) {
  infile <- file.path(INPUT_DIR, paste0(region, "_module_celltype_gene_fraction.tsv"))

  if (!file.exists(infile)) {
    stop("Input file not found: ", infile)
  }

  message("Reading: ", infile)

  read_tsv(infile, show_col_types = FALSE) %>%
    mutate(region = region)
}

all_fraction <- map_dfr(regions_needed, read_one_region)

required_cols <- c("module", "celltype", "n_genes", "module_total", "fraction")
missing_cols <- setdiff(required_cols, colnames(all_fraction))
if (length(missing_cols) > 0) {
  stop("Missing required columns: ", paste(missing_cols, collapse = ", "))
}

# ============================================================


In [ ]:
# 3. Keep the currently selected cross-cell-type modules
# ============================================================

selected_raw <- all_fraction %>%
  inner_join(
    SELECTED_MODULES %>% select(region, module, region_module),
    by = c("region", "module")
  )

# Ensure a percent column is always available even if an input file contains
# only the fraction column.
if (!"percent" %in% names(selected_raw)) {
  selected_raw <- selected_raw %>% mutate(percent = fraction * 100)
}

# Check that every requested module was found.
found_modules <- selected_raw %>% distinct(region_module)
missing_modules <- setdiff(MODULE_ORDER, found_modules$region_module)
if (length(missing_modules) > 0) {
  stop(
    "Requested module(s) not found in the input files: ",
    paste(missing_modules, collapse = ", ")
  )
}

# No duplicated cell-type row should occur within a module.
duplicates <- selected_raw %>%
  count(region_module, celltype) %>%
  filter(n > 1)

if (nrow(duplicates) > 0) {
  print(duplicates)
  stop("Duplicated region_module/celltype rows detected.")
}

# Keep one module_total per module and verify it is internally consistent.
module_total_check <- selected_raw %>%
  group_by(region_module) %>%
  summarise(
    n_distinct_module_total = n_distinct(module_total),
    module_total = first(module_total),
    .groups = "drop"
  )

if (any(module_total_check$n_distinct_module_total != 1)) {
  print(module_total_check %>% filter(n_distinct_module_total != 1))
  stop("A module has inconsistent module_total values across cell types.")
}

module_totals <- module_total_check %>%
  select(region_module, module_total)

# ============================================================


In [ ]:
# This preserves zero-count cell types if any selected module is missing a row.
plot_df <- tidyr::crossing(
  region_module = MODULE_ORDER,
  celltype = CELLTYPE_ORDER
) %>%
  left_join(
    SELECTED_MODULES %>%
      select(region_module, region, module),
    by = "region_module"
  ) %>%
  left_join(
    selected_raw %>%
      select(
        region_module,
        celltype,
        n_genes,
        fraction,
        any_of("percent")
      ),
    by = c("region_module", "celltype")
  ) %>%
  # Add module_total only once here
  left_join(
    module_totals,
    by = "region_module"
  ) %>%
  mutate(
    n_genes = dplyr::coalesce(as.numeric(n_genes), 0),
    fraction = dplyr::coalesce(as.numeric(fraction), 0),

    # If percent is NA, calculate it from fraction
    percent = dplyr::coalesce(
      as.numeric(percent),
      fraction * 100
    ),

    module_total = as.numeric(module_total)
  )


# Ordered plotting coordinates
x_map <- tibble(
  region_module = MODULE_ORDER,
  x = seq_along(MODULE_ORDER)
)

In [ ]:
head(plot_df) 

In [ ]:

y_map <- tibble(
  celltype = CELLTYPE_ORDER,
  y = rev(seq_along(CELLTYPE_ORDER))
)

plot_df <- plot_df %>%
  left_join(x_map, by = "region_module") %>%
  left_join(y_map, by = "celltype") %>%
  mutate(
    region_module = factor(region_module, levels = MODULE_ORDER),
    celltype = factor(celltype, levels = CELLTYPE_ORDER)
  )

# ============================================================


In [ ]:
# 5. Validation
# ============================================================

validation <- plot_df %>%
  group_by(region_module) %>%
  summarise(
    region = first(region),
    module = first(module),
    module_total = first(module_total),
    sum_n_genes = sum(n_genes, na.rm = TRUE),
    sum_fraction = sum(fraction, na.rm = TRUE),
    count_matches_total = sum_n_genes == module_total,
    fraction_sums_to_one = abs(sum_fraction - 1) < 1e-6,
    .groups = "drop"
  )

print(validation)

if (any(!validation$count_matches_total)) {
  warning("For at least one module, sum(n_genes) != module_total. See validation table.")
}

if (any(!validation$fraction_sums_to_one)) {
  warning("For at least one module, cell-type fractions do not sum to 1. See validation table.")
}

fraction_formula_check <- plot_df %>%
  filter(module_total > 0) %>%
  mutate(expected_fraction = n_genes / module_total,
         difference = abs(fraction - expected_fraction))

if (any(fraction_formula_check$difference > 1e-8, na.rm = TRUE)) {
  warning("At least one reported fraction differs from n_genes/module_total by > 1e-8.")
}

# Save the selected plotting table and validation table.
write_tsv(
  plot_df %>%
    transmute(
      region,
      module,
      region_module = as.character(region_module),
      celltype = as.character(celltype),
      n_genes,
      module_total,
      fraction,
      percent
    ),
  file.path(OUT_DIR, "selected_cross_celltype_module_gene_fraction.tsv")
)

write_tsv(
  validation %>% mutate(region_module = as.character(region_module)),
  file.path(OUT_DIR, "selected_cross_celltype_module_fraction_validation.tsv")
)

# ============================================================


In [ ]:
# 6. Plot
# ============================================================

# Module totals for x-axis labels.
module_label_df <- plot_df %>%
  distinct(region_module, x, module_total) %>%
  arrange(x) %>%
  mutate(
    x_label = paste0(as.character(region_module), "\n(n=", module_total, ")")
  )

# Region-colored strip above each module column.
strip_df <- plot_df %>%
  distinct(region_module, region, x) %>%
  arrange(x) %>%
  mutate(strip_y = max(y_map$y) + 0.78)

p <- ggplot() +
  # Light guide lines make the matrix structure clear without boxing the plot.
  geom_hline(
    yintercept = y_map$y,
    color = "grey92",
    linewidth = 0.45
  ) +
  geom_vline(
    xintercept = x_map$x,
    color = "grey94",
    linewidth = 0.45
  ) +

  # Brain-region annotation strip.
  geom_tile(
    data = strip_df,
    aes(x = x, y = strip_y, fill = region),
    width = 0.82,
    height = 0.22,
    color = NA
  ) +

  # Bubble area = gene fraction; color = cell type.
  geom_point(
    data = plot_df,
    aes(x = x, y = y, size = fraction, color = celltype),
    alpha = 0.92
  ) +

  # Use smaller labels inside bubbles with small gene fractions.
  geom_text(
    data = plot_df %>% filter(n_genes > 0, fraction >= 0.03),
    aes(x = x, y = y, label = n_genes),
    color = "white",
    fontface = "bold",
    size = 3.3
  ) +
  geom_text(
    data = plot_df %>% filter(n_genes > 0, fraction < 0.03),
    aes(x = x, y = y, label = n_genes),
    color = "white",
    fontface = "bold",
    size = 2.2
  ) +

  scale_color_manual(
    values = CELLTYPE_COLORS,
    limits = CELLTYPE_ORDER,
    drop = FALSE,
    guide = "none"
  ) +
  scale_fill_manual(
    values = REGION_COLORS,
    name = "Region"
  ) +
  scale_size_area(
    max_size = 18,
    name = "Gene fraction",
    labels = label_percent(accuracy = 1)
  ) +
  scale_x_continuous(
    breaks = module_label_df$x,
    labels = module_label_df$x_label,
    limits = c(0.45, length(MODULE_ORDER) + 0.55),
    expand = expansion(mult = 0)
  ) +
  scale_y_continuous(
    breaks = y_map$y,
    labels = y_map$celltype,
    limits = c(0.45, max(y_map$y) + 1.10),
    expand = expansion(mult = 0)
  ) +
  labs(
    x = NULL,
    y = NULL#,
    #title = "Cell-type gene composition of selected cross-cell-type modules",
    #subtitle = "Bubble area = fraction of module genes; number inside bubble = cell-type gene count; x-axis n = total module genes"
  ) +
  guides(
    fill = guide_legend(order = 1, override.aes = list(size = 5)),
    size = guide_legend(order = 2)
  ) +
  coord_cartesian(clip = "off") +
  theme_classic(base_size = 15) +
  theme(
    plot.title = element_text(size = 18, face = "bold", hjust = 0),
    plot.subtitle = element_text(size = 12.5, hjust = 0, margin = margin(b = 12)),
    axis.text.x = element_text(size = 11.5, face = "bold", angle = 90, hjust = 1, vjust = 1),
    axis.text.y = element_text(size = 13, face = "bold", color = "black"),
    axis.ticks = element_blank(),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11),
    plot.margin = margin(t = 10, r = 18, b = 10, l = 10)
  )

print(p)

# ============================================================


In [ ]:
# 7. Save figure
# ============================================================

ggsave(
  filename = file.path(OUT_DIR, "selected_cross_celltype_module_gene_fraction_bubbleplot.pdf"),
  plot = p,
  width = 8,
  height = 5.5,
  units = "in"
)

# ggsave(
#   filename = file.path(OUT_DIR, "selected_cross_celltype_module_gene_fraction_bubbleplot.png"),
#   plot = p,
#   width = 7,
#   height = 4.8,
#   units = "in",
#   dpi = 300
# )

message("Done. Outputs saved in: ", OUT_DIR)
